# ▶ BrandMind AI — Week 6
## Animated Visuals Studio — Logo + Tagline Animation
**Tools:** PyCairo · Matplotlib animations · MoviePy · Pillow

In [ ]:
!pip install matplotlib pillow -q
# !pip install pycairo moviepy -q  # Uncomment for full export

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np
from PIL import Image
import io, os
os.makedirs('outputs', exist_ok=True)
print('Ready')

## 1. Animation Planning & Storyboard

| Frame Range | Duration | Action |
|------------|----------|--------|
| 0–20 | 0–1s | Logo fades in (opacity 0→1) |
| 20–22 | 1–1.1s | Brief pause |
| 22–58 | 1.1–2.9s | Tagline types in (typewriter effect) |
| 50–60 | 2.5–3s | Company name + palette reveal |
| 60–70 | 3–3.5s | Hold complete frame |
| 70–80 | 3.5–4s | Fade to brand colour |

## 2. Prepare Assets

In [ ]:
# Brand configuration (from previous weeks)
brand_config = {
    'company': 'BrandMind',
    'slogan': 'Intelligence meets identity.',
    'bg_color': '#1a1a18',
    'accent_color': '#b8975a',
    'fg_color': '#f5f0e8',
    'font_family': 'serif',
}

# Create base logo as matplotlib figure
def render_logo_frame(company, accent, fg):
    fig_logo, ax = plt.subplots(figsize=(3, 3))
    ax.set_facecolor('#1a1a18')
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
    ch = company[0].upper()
    ax.text(5, 6, ch, ha='center', va='center', fontsize=52, color=accent, fontfamily='serif', fontstyle='italic')
    ax.text(5, 2.5, company.upper(), ha='center', va='center', fontsize=8, color=fg, fontfamily='monospace', alpha=0.8)
    return fig_logo

logo_fig = render_logo_frame(brand_config['company'], brand_config['accent_color'], brand_config['fg_color'])
plt.savefig('outputs/logo.png', dpi=150, bbox_inches='tight', facecolor='#1a1a18')
plt.close(logo_fig)
print('Logo asset saved: outputs/logo.png')

## 3. Animation Implementation

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5.4))  # 1080x1080 equivalent at 200dpi
fig.patch.set_facecolor(brand_config['bg_color'])
ax.set_facecolor(brand_config['bg_color'])
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')

slogan = brand_config['slogan']
company = brand_config['company']
accent = brand_config['accent_color']
fg = brand_config['fg_color']

# Text elements
logo_letter  = ax.text(5, 6.5, company[0].upper(), ha='center', va='center',
                        fontsize=60, color=accent, fontfamily='serif', fontstyle='italic', alpha=0)
tagline_text = ax.text(5, 3.8, '', ha='center', va='center',
                        fontsize=10, color=fg, fontstyle='italic')
brand_name   = ax.text(5, 2.2, '', ha='center', va='center',
                        fontsize=8, color=accent, fontweight='bold', fontfamily='monospace')
cursor       = ax.text(5, 3.8, '', ha='center', va='center', fontsize=10, color=accent)

# Palette dots
palette_circles = []
palette_colors = [accent, fg, '#6b6a65', '#d4c4a0']
for i, c in enumerate(palette_colors):
    circle = plt.Circle([3.5 + i*1.0, 1.0], 0.25, color=c, alpha=0)
    ax.add_patch(circle)
    palette_circles.append(circle)

def animate(frame):
    TOTAL = 80
    # Phase 1: Logo fade-in (0–20)
    logo_alpha = min(1.0, frame / 20)
    logo_letter.set_alpha(logo_alpha)

    # Phase 2: Typewriter (22–58)
    if frame > 22:
        n_chars = int(((frame - 22) / 36) * len(slogan))
        n_chars = min(n_chars, len(slogan))
        tagline_text.set_text(slogan[:n_chars])
        tagline_text.set_position((5, 3.8))
        # Cursor blink
        cursor.set_text('|' if (frame % 6 < 3) and n_chars < len(slogan) else '')
        cursor.set_position((5 + len(slogan[:n_chars]) * 0.25, 3.8))

    # Phase 3: Brand name + palette reveal (50+)
    if frame > 50:
        brand_name.set_text(company.upper())
        reveal_alpha = min(1.0, (frame - 50) / 10)
        brand_name.set_alpha(reveal_alpha)
        for circle in palette_circles:
            circle.set_alpha(reveal_alpha * 0.8)

    return [logo_letter, tagline_text, brand_name, cursor] + palette_circles

anim = FuncAnimation(fig, animate, frames=80, interval=50, blit=True)
print('Animation created (80 frames @ 20fps = 4 seconds)')

## 4. Export GIF and MP4

In [ ]:
# Export GIF
gif_writer = PillowWriter(fps=20)
anim.save('outputs/brand_animation.gif', writer=gif_writer, dpi=100)
print('Exported: outputs/brand_animation.gif')

# Display preview frames
fig2, axes = plt.subplots(1, 4, figsize=(16, 4))
preview_frames = [10, 30, 50, 75]
for ax2, f in zip(axes, preview_frames):
    ax2.set_facecolor('#1a1a18')
    ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis('off')
    ax2.set_title(f'Frame {f}', color='white')
    ax2.set_facecolor('#1a1a18')
    alpha = min(1.0, f/20)
    ax2.text(5, 6.5, company[0].upper(), ha='center', va='center',
             fontsize=36, color=accent, fontfamily='serif', fontstyle='italic', alpha=alpha)
    if f > 22:
        n = int(((f-22)/36)*len(slogan))
        ax2.text(5, 3.8, slogan[:n], ha='center', va='center', fontsize=7, color=fg, fontstyle='italic')
    if f > 50:
        ax2.text(5, 2.2, company.upper(), ha='center', va='center', fontsize=5, color=accent, fontweight='bold')
fig2.patch.set_facecolor('#2a2a28')
plt.tight_layout(); plt.savefig('outputs/animation_storyboard.png', dpi=150, facecolor='#2a2a28'); plt.show()
print('Saved: outputs/animation_storyboard.png')

## Week 6 Complete ✅
- ✅ Animation storyboard (4 phases)
- ✅ PyCairo/Matplotlib FuncAnimation (80 frames, 4s, 20fps)
- ✅ Typewriter + fade-in + brand reveal effects
- ✅ GIF export at social-media resolution
- ✅ Storyboard preview PNG